The libraries are considering the native databricks enviroment at 18-03-2026

In [ ]:
#%pip install openpyxl  #install if necessary
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.types import *

BRONZE

In [ ]:
# Read from Volume (not dbfs:/)
pandas_df = pd.read_excel(
    '/Workspace/Projects/Feature Store + Training Pipeline/online_retail_II.xlsx',
    engine='openpyxl',
    dtype_backend='numpy_nullable'  # avoids object dtype issues
)

# Convert to Spark DataFrame
raw_df = spark.createDataFrame(pandas_df)

# Write to Delta
raw_df.write.format('delta') \
    .mode('overwrite') \
    .option('delta.columnMapping.mode', 'name') \
    .saveAsTable('ml1.ml_project.bronze_transactions')

SILVER

In [ ]:
# Clean: drop nulls on key fields, remove returns (negative quantity)
silver_df = spark.table('ml1.ml_project.bronze_transactions') \
    .dropna(subset=['Customer ID', 'Invoice', 'Price']) \
    .filter(F.col('Quantity') > 0) \
    .filter(F.col('Price') > 0) \
    .withColumn('revenue', F.col('Quantity') * F.col('Price')) \
    .withColumn('invoice_date', F.to_timestamp('InvoiceDate'))

silver_df.write.format('delta') \
    .mode('overwrite') \
    .option('delta.columnMapping.mode', 'name') \
    .saveAsTable('ml1.ml_project.silver_transactions')


GOLD

In [ ]:
# Aggregate to customer level — this is what feeds the feature store
snapshot_date = '2011-09-01'  # Use a fixed snapshot to simulate a point-in-time view

gold_df = spark.table('ml1.ml_project.silver_transactions') \
    .filter(F.col('invoice_date') < snapshot_date) \
    .groupBy('Customer ID') \
    .agg(
        F.max('invoice_date').alias('last_purchase_date'),
        F.countDistinct('Invoice').alias('total_orders'),
        F.sum('revenue').alias('total_revenue'),
        F.avg('revenue').alias('avg_order_value')
    ) \
    .withColumnRenamed('Customer ID', 'customer_id')

gold_df.write.format('delta') \
    .mode('overwrite') \
    .option('delta.columnMapping.mode', 'name') \
    .saveAsTable('ml1.ml_project.gold_customer_summary')

print(f'Gold table row count: {gold_df.count()}')

In [ ]:
%sql
select * from ml1.ml_project.gold_customer_summary 